In [ ]:
import os
import itertools
import pandas as pd
import cv2
import glob
import json

cats = ["bottle", "bowl", "chair", "cup", 
        "door", "spoon", "table", "window"]

df_data = pd.read_csv("../../RGB_hist/data/sub_data_release.csv") 

# build mapping from frame_id to a list of its categories
frame_to_category = df_data.groupby('frame_id')['obj'].apply(list).to_dict()

image_folder = "/path/to/egocentric_images/" # these will be released to Databrary
histograms = {}

# just a JSON file contain top and non top filenames
with open("../../instance_filenames.json", "r") as f:
    top_nonTop_instances = json.load(f)
    
for image_path in glob.glob(os.path.join(image_folder, "*.jpg")):
    filename = os.path.basename(image_path)
    image = cv2.imread(image_path)
    if image is None:
        continue
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    hist = cv2.calcHist([image_rgb], [0, 1, 2], None, [8, 8, 8],
                        [0, 256, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()
    histograms[filename] = hist


rows = []
# get unique subjects from the original dataframe
subjects = df_data['subj'].unique()

for subj in subjects:
    instances = top_nonTop_instances[subj]
    for cat in cats:
        all_cats = []
        top_cats = [] # debug
        other_cats = [] # debug
        all_cats.extend(instances['all'][cat]) # get top or non top for all cats combined
        top_cats.extend(instances['top'][cat]) # debug
        other_cats.extend(instances['non-top'][cat]) # debug

        # get all unique frame_ids for the current subject
        subj_frames = df_data[df_data['subj'] == subj]['frame_id'].unique()
        
        valid_frames = []
        for f in subj_frames:
            if f in histograms and f in all_cats:# and (f in top_cats or f in other_cats): # if doing all to all you can comment out all cats
                valid_frames.append(f)
            else:
                pass
                # print(f)
                # print(len(frame_ids))
                # print(f in jpgs)
        
        if len(valid_frames) < 10:
            # if type_comparison != 'all':
            continue
        
        pairs = list(itertools.combinations(valid_frames, 2))

        for img1, img2 in pairs:
            # get lists of categories for each image
            cat_list1 = set(frame_to_category.get(img1, []))
            cat_list2 = set(frame_to_category.get(img2, []))

            if len(cat_list1) == 0 or len(cat_list2) == 0:
                raise
            
            # compute histogram correlation once per image pair
            hist_corr = cv2.compareHist(histograms[img1], histograms[img2], cv2.HISTCMP_CORREL)

            rows.append(
                {
                "subj": subj,
                "img1": img1,
                "img2": img2,
                "category": f"{cat}_{cat}",
                "valid": (cat == cat),
                "hist_corr": hist_corr
                }
            )

df_pairs = pd.DataFrame(rows)
print(df_pairs [df_pairs['valid'] == True].shape)
print(df_pairs.shape)
df_pairs.to_csv("../../RGB_hist/all2all_corr_bysubj.csv", index=False) 

In [ ]:
import pandas as pd
###############     # infant cats    ###############
chosen_cats = ["bottle", "bowl", "chair", "cup", 
                "door", "spoon", "table", "window"]

df_data = pd.read_csv("../../RGB_hist/all2all_corr_bysubj.csv")  # Columns: frame_id, obj
# print(df_data.head())
# just splitting into
totall = 0
for cat in chosen_cats:
    mask = df_data["category"] == f"{cat}_{cat}"
    maskb = df_data["valid"] == True
    df = df_data [ mask & maskb ].copy()
    totall += df.shape[0]
    print(df.shape)
    df.to_csv(f"../../RGB_hist/all2all_corr_bysubj_forRplots_{cat}.csv", index=False, header=None)
    del df

print(totall)